# Phân tích dữ liệu 32 trạm sau preprocessing
Mỗi cell vẽ 1 feature cho tất cả 32 trạm.

In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load all stations
files = sorted(glob.glob('data/clean/station_*.csv'),
               key=lambda x: int(os.path.basename(x).replace('station_','').replace('.csv','')))
stations = {}
for f in files:
    sid = int(os.path.basename(f).replace('station_','').replace('.csv',''))
    df = pd.read_csv(f, parse_dates=['time'])
    stations[sid] = df
print(f'Found {len(stations)} station files.')

In [ ]:
def plot_feature(feature, ylabel=None, figsize=(28, 24)):
    """Vẽ time series của 1 feature cho tất cả 32 trạm."""
    n_stations = len(stations)
    ncols = 4
    nrows = (n_stations + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True)
    axes = axes.flatten()
    for i, (sid, df) in enumerate(sorted(stations.items())):
        ax = axes[i]
        if feature in df.columns:
            ax.plot(df['time'], df[feature], linewidth=0.5)
        ax.set_title(f'Station {sid}', fontsize=10)
        ax.set_ylabel(ylabel or feature, fontsize=8)
        ax.tick_params(axis='both', labelsize=7)
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    fig.suptitle(f'{feature} — All Stations', fontsize=16, y=1.01)
    plt.tight_layout()
    plt.show()

## AQI

In [ ]:
plot_feature('aqi', ylabel='AQI')

## PM2.5

In [ ]:
plot_feature('pm25', ylabel='PM2.5 (µg/m³)')

## PM10

In [ ]:
plot_feature('pm10', ylabel='PM10 (µg/m³)')

## CO

In [ ]:
plot_feature('co', ylabel='CO (µg/m³)')

## MA PM2.5 (24h)

In [ ]:
plot_feature('ma_pm25_24', ylabel='MA PM2.5 24h')

## Relative Humidity (RH)

In [ ]:
plot_feature('rh', ylabel='RH (%)')

## Dew Point

In [ ]:
plot_feature('dewpt', ylabel='Dew Point (°C)')

## Temperature

In [ ]:
plot_feature('temp', ylabel='Temperature (°C)')

## Precipitation

In [ ]:
plot_feature('precip', ylabel='Precip (mm)')

## Wind Speed

In [ ]:
plot_feature('wind_spd', ylabel='Wind Speed (m/s)')

## Distribution Overview

In [ ]:
# Histogram cho các feature chính
features_to_hist = ['aqi', 'pm25', 'pm10', 'co', 'rh', 'dewpt', 'temp', 'precip', 'wind_spd']
all_data = pd.concat(stations.values(), ignore_index=True)

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()
for i, feat in enumerate(features_to_hist):
    ax = axes[i]
    data = all_data[feat].dropna()
    ax.hist(data, bins=80, edgecolor='white', alpha=0.7)
    ax.set_title(feat, fontsize=12)
    ax.set_ylabel('Count')
    # Show basic stats
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1, label=f'median={data.median():.1f}')
    ax.legend(fontsize=8)
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
fig.suptitle('Distributions of Key Features (All Stations)', fontsize=16)
plt.tight_layout()
plt.show()

## Missing Data Summary

In [ ]:
# Missing data per station
summary = []
for sid, df in sorted(stations.items()):
    row = {'station': sid, 'rows': len(df)}
    for col in ['aqi','pm25','pm10','co','rh','dewpt','temp','precip','wind_spd']:
        if col in df.columns:
            row[f'{col}_missing%'] = round(df[col].isnull().mean() * 100, 2)
    summary.append(row)
summary_df = pd.DataFrame(summary).set_index('station')
print(summary_df.to_string())
print(f'\nAverage rows per station: {summary_df["rows"].mean():.0f}')